# 02 — FRED Mortgage Rate Ingest & Clean

**Purpose.** Download `MORTGAGE30US` (the 30-year fixed mortgage rate) from FRED, resample it from weekly to monthly, and load the result into Postgres as `mortgage_monthly`.

This is the only FRED series in scope for this project (see `docs/phase0_framing.md`'s Field Scope — no other FRED series are pulled). It's a standalone ingest: no join to Zillow happens here, that's `03_join_sql_load.ipynb`.


## 1. Setup

In [1]:
import os
import pandas as pd
from sqlalchemy import text

from housing_risk.config import get_engine

pd.set_option("display.max_columns", 20)


In [2]:
# --- Database connection ---
# Shared with every notebook in this project via src/housing_risk/config.py,
# so credentials and cutoff dates only have to be defined once.
engine = get_engine()

with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).scalar())


PostgreSQL 15.18 (Homebrew) on aarch64-apple-darwin25.4.0, compiled by Apple clang version 21.0.0 (clang-2100.0.123.102), 64-bit


## 2. Schema

`mortgage_monthly` mirrors `zhvi_monthly`'s shape: one row per month, national grain (no county key — `MORTGAGE30US` is a single national series applied uniformly to every county in a given month during the join step, not during ingest).


In [3]:
DDL = """
CREATE TABLE IF NOT EXISTS mortgage_monthly (
    month           DATE NOT NULL,
    mortgage_rate   DOUBLE PRECISION,
    PRIMARY KEY (month)
);
"""

with engine.begin() as conn:
    conn.execute(text(DDL))


## 3. Download

FRED publishes `MORTGAGE30US` as a CSV via its `fredgraph.csv` endpoint — no API key required for a straight CSV pull. If your team has a FRED API key set up already (for rate-limit headroom or other series later), swap this for the `fredapi` package instead; for this single series a direct CSV request is enough.


In [4]:
import requests

FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

RAW_DIR = "../data/raw/fred"
os.makedirs(RAW_DIR, exist_ok=True)
raw_path = os.path.join(RAW_DIR, "mortgage30us.csv")

if not os.path.exists(raw_path):
    resp = requests.get(FRED_URL, timeout=60)
    resp.raise_for_status()
    with open(raw_path, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content):,} bytes to {raw_path}")
else:
    print(f"Using cached file at {raw_path}")

rate_weekly = pd.read_csv(raw_path)
rate_weekly.columns = ["date", "mortgage_rate"]
rate_weekly["date"] = pd.to_datetime(rate_weekly["date"])
rate_weekly["mortgage_rate"] = pd.to_numeric(rate_weekly["mortgage_rate"], errors="coerce")
rate_weekly = rate_weekly.dropna(subset=["mortgage_rate"])
rate_weekly.head()


Using cached file at ../data/raw/fred/mortgage30us.csv


,date,mortgage_rate
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


## 4. Resample weekly → monthly

`MORTGAGE30US` is published weekly (Thursdays). Resample to monthly by taking each month's mean of the weekly readings, then align to month-end to match Zillow's monthly grain, which is what Phase 1's join step depends on.


In [5]:
rate_monthly = (
    rate_weekly.set_index("date")["mortgage_rate"]
    .resample("ME")
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"date": "month", "mean": "mortgage_rate", "count": "n_weeks"})
)

rate_monthly = rate_monthly.dropna(subset=["mortgage_rate"])
rate_monthly.head()

,month,mortgage_rate,n_weeks
0,1971-04-30,7.3100,5
1,1971-05-31,7.4250,4
2,1971-06-30,7.5300,4
3,1971-07-31,7.6040,5
4,1971-08-31,7.6975,4


## 5. Load to Postgres

In [6]:
# Clear existing rows first so this cell is safe to re-run (e.g. after changing
# the resample convention, or just re-running the notebook top to bottom) --
# to_sql(if_exists="append") alone raises a UniqueViolation on `month` otherwise.
with engine.begin() as conn:
    conn.execute(text("DELETE FROM mortgage_monthly;"))

rate_monthly.to_sql(
    "mortgage_monthly",
    engine,
    if_exists="append",
    index=False,
)

with engine.connect() as conn:
    loaded = conn.execute(text("SELECT COUNT(*) FROM mortgage_monthly;")).scalar()
print(f"mortgage_monthly now has {loaded:,} rows")


mortgage_monthly now has 666 rows


## 6. Sanity checks

In [7]:
assert loaded == len(rate_monthly), "Row count in Postgres doesn't match the cleaned frame — check for a partial or duplicate load."

with engine.connect() as conn:
    sample = pd.read_sql(
        "SELECT * FROM mortgage_monthly ORDER BY month LIMIT 5;", conn
    )
sample


,month,mortgage_rate,n_weeks
0,1971-04-30 00:00:00,7.3100,5
1,1971-05-31 00:00:00,7.4250,4
2,1971-06-30 00:00:00,7.5300,4
3,1971-07-31 00:00:00,7.6040,5
4,1971-08-31 00:00:00,7.6975,4
